**Sửa tổng hợp sau resume:** chuyển cột điểm về số và đọc đúng cờ True/False trước khi tính trung bình. Hỗ trợ checkpoint mới có 8.995 ảnh thành công; vòng lặp sẽ chỉ thử lại 5 ảnh lỗi.

**Sửa lỗi ảnh nhỏ:** truyền rõ `images_kwargs={"input_data_format": "channels_last"}` sau khi chuyển RGB. Lỗi `mean must have 1 elements ... got 3` với ảnh `(1, 1, 3)` không còn do suy nhầm chiều kênh. Nếu phiên cũ đang chạy, để phiên đó hoàn tất rồi dùng output mới nhất để thử lại các ảnh lỗi bằng bản này.

# Florence-2-base — Resume OPRB test evaluation

Bản tiếp tục từ checkpoint ít nhất **5.790 ảnh thành công**. Gắn **output của lần chạy Florence trước** qua **Add Input** rồi chạy notebook. Cell sau Configuration sẽ tự khôi phục CSV + cấu hình vào `/kaggle/working`, giữ nguyên checkpoint nếu đã có tiến độ mới, và dừng nếu gắn nhầm bản thử 10 ảnh. Hỗ trợ cả CSV và CSV.gz.

Mặc định chọn cả hai nhóm, `MAX_SAMPLES_PER_CATEGORY = None`, ngân sách 10 giờ/phiên. Giữ nguyên FP16, 1.024 token, 3 beams và revision mô hình của checkpoint. Kiểm tra phiên bản Transformers/PyTorch theo thông tin in ra; không bỏ qua lỗi khác cấu hình ở Mục 9.

**Save & Run All** tạo phiên chạy riêng, nên phải giữ cell khôi phục trong notebook; chỉ chép file một lần bằng tay trong phiên tương tác là chưa đủ. Theo dõi Mục 9: `Reusing 5,790 successful selected pages; 3,210 remaining.` Hai ảnh lỗi trước sẽ được thử lại.

# OPRB test-set benchmark — Florence-2-base, standard decoding

Evaluate the pretrained **Florence-2-base** vision-language model on occluded document images, without training or fine-tuning. This notebook evaluates **one** model; the assignment calls for a comparison of at least five models.

| Component | Choice |
|---|---|
| Dataset | [OPRB](https://huggingface.co/datasets/kpurkayastha/OPRB), test split only |
| Allowed model input | Occluded images from `test/text/img/` and `test/nontext/img/` |
| Ground truth | Matching `test/*/gt/` annotations, used only for scoring |
| Model | [`florence-community/Florence-2-base`](https://huggingface.co/florence-community/Florence-2-base), the Transformers-compatible checkpoint |
| Precision | Float16 on one CUDA GPU; no quantization |
| OCR task | `<OCR>`, checkpoint defaults: 3 beams and no repeated token trigrams |
| Metrics | The same per-image CER, WER, bag-of-words token precision/recall/F1, and latency as the other OCR benchmark notebooks |
| Outputs | Per-image CSV, grouped CSVs, JSON settings, plots, and ZIP |

**Kaggle steps:** enable a GPU and Internet, then run the code cells in order. Start with `MAX_SAMPLES_PER_CATEGORY = 5` (10 pages total). After verifying the outputs and timing, set it to `None` if evaluating the full 9,000-page test set. Confirm with your instructor whether a smaller common subset is permitted.

The notebook reuses `/kaggle/working/oprb_test` when the downloaded files are present. Results go to `/kaggle/working/oprb_florence_standard_results`, in a dedicated results directory. Back up the checkpoint CSV and `checkpoint_config.json` before ending a Kaggle session; restore both into the results directory before resuming in a fresh session.

**Fair evaluation:** clean images (`img_gt/`) and occluded annotations (`annots/`) are not downloaded. GT text is never passed to the model. The processor uses its checkpoint's default image preprocessing; the effective size is printed and saved. The checkpoint supports at most 1,024 decoder positions, so long transcripts may be truncated and are flagged.

Implementation reference: [Transformers Florence-2 documentation](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/florence2).

**Trial revision:** The first fast trial used greedy decoding with the repetition guard disabled. Its output repeated text/numbers and reached the token limit on 9 of 10 pages. This version restores the checkpoint's published decoding defaults (`num_beams=3`, `no_repeat_ngram_size=3`, `early_stopping=True`). Run the same 10-page trial again; accuracy and speed of this revision have not been measured yet. The repetition rule can also suppress legitimate repeated token sequences, so record these settings in the report.

Defaults source: [checkpoint generation_config.json](https://huggingface.co/florence-community/Florence-2-base/blob/main/generation_config.json).


## 1. Install dependencies

In [ ]:
%pip -q install "transformers==4.57.6" "accelerate>=1.0" "huggingface_hub>=0.34,<1" "jiwer>=3.0" "pandas>=2.0" matplotlib "Pillow>=10,<13" tqdm

# If this changes a library already imported in this kernel, restart the kernel
# and continue from Section 2. Florence does not need bitsandbytes or flash-attn.


## 2. Imports and GPU check

In [ ]:
import gc, json, random, re, shutil, tarfile, time, unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from PIL import Image
from IPython.display import display
from huggingface_hub import hf_hub_download
from jiwer import cer, wer
from tqdm import tqdm
import matplotlib.pyplot as plt

assert torch.cuda.is_available(), "CUDA GPU required. Enable a GPU in Kaggle session options."
print('GPU:', torch.cuda.get_device_name(0))
print('Torch:', torch.__version__)
print('CUDA:', torch.version.cuda)

## 3. Configuration — edit these settings first

In [ ]:
REPO_ID = "kpurkayastha/OPRB"
MODEL_ID = "florence-community/Florence-2-base"
DATA_ROOT = Path("/kaggle/working/oprb_test")
OUTPUT_DIR = Path("/kaggle/working/oprb_florence_standard_results")

CATEGORIES = ["text", "nontext"]
MAX_SAMPLES_PER_CATEGORY = None  # Resume all 9,000 selected pages.
RANDOM_SEED = 42

# Use the checkpoint's default image preprocessing (printed in Section 7).
# This decoder has 1,024 learned positions; do not exceed that token budget.
MAX_NEW_TOKENS = 1024
NUM_BEAMS = 3  # Published checkpoint default.
NO_REPEAT_NGRAM_SIZE = 3
EARLY_STOPPING = True
NORMALIZE_CASE = True

# Stop the evaluation loop gracefully so it can export before a session expires.
# This budget starts in Section 9; keep time for installation/downloads in mind.
MAX_EVAL_HOURS = 10.0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
assert NUM_BEAMS >= 1
assert NO_REPEAT_NGRAM_SIZE >= 0
assert MAX_EVAL_HOURS is None or MAX_EVAL_HOURS > 0
print('Model:', MODEL_ID)
print('Categories:', CATEGORIES)
print('Maximum samples per category:', MAX_SAMPLES_PER_CATEGORY)
print('Results directory:', OUTPUT_DIR)


### Khôi phục checkpoint trước khi tải dữ liệu và mô hình
Gắn đúng một output Florence cũ trong Input. Cell dưới không thay thế checkpoint đã có trong OUTPUT_DIR.


In [ ]:
# Chạy cell này trong cả phiên tương tác và Save Version -> Save & Run All.
# Gắn output của notebook Florence cũ qua Add Input trước khi chạy.
import csv
import gzip
import json
import shutil
import sys

csv.field_size_limit(min(sys.maxsize, 2**31 - 1))
resume_csv = OUTPUT_DIR / "per_image_results.csv"
resume_json = OUTPUT_DIR / "checkpoint_config.json"

if resume_csv.exists() != resume_json.exists():
    raise RuntimeError("OUTPUT_DIR thiếu một trong hai file checkpoint. Cần CSV và checkpoint_config.json đi cùng nhau.")

if resume_csv.exists():
    resume_source = OUTPUT_DIR
    print("Dùng checkpoint đang có trong OUTPUT_DIR; giữ nguyên tiến độ mới hơn.")
else:
    candidates = []
    for config_path in Path("/kaggle/input").rglob("checkpoint_config.json"):
        try:
            settings = json.loads(config_path.read_text(encoding="utf-8"))
        except (OSError, ValueError):
            continue
        folder = config_path.parent
        has_results = (folder / "per_image_results.csv").is_file() or (folder / "per_image_results.csv.gz").is_file()
        if settings.get("model") == MODEL_ID and has_results:
            candidates.append(folder)
    if len(candidates) != 1:
        raise RuntimeError(
            f"Tìm thấy {len(candidates)} checkpoint Florence trong Input: {candidates}. "
            "Hãy gắn đúng một output Florence đã chạy 5.790 ảnh, có CSV và checkpoint_config.json."
        )
    resume_source = candidates[0]

resume_config = json.loads((resume_source / "checkpoint_config.json").read_text(encoding="utf-8"))
if resume_config.get("model") != MODEL_ID or not resume_config.get("model_revision"):
    raise RuntimeError("Checkpoint không đúng mô hình Florence hoặc thiếu model_revision.")

plain_csv = resume_source / "per_image_results.csv"
input_csv = plain_csv if plain_csv.is_file() else resume_source / "per_image_results.csv.gz"
previous_checkpoint = pd.read_csv(input_csv, engine="python", keep_default_na=False)
if previous_checkpoint.duplicated(["category", "sample_id"]).any():
    raise RuntimeError("Checkpoint chứa ID ảnh trùng.")
if not previous_checkpoint["model_id"].eq(MODEL_ID).all():
    raise RuntimeError("CSV chứa kết quả của mô hình khác.")
resume_successes = int(previous_checkpoint["status"].eq("ok").sum())
if resume_successes < 5790:
    raise RuntimeError(f"Checkpoint chỉ có {resume_successes:,} ảnh thành công. Hãy chọn output có ít nhất 5.790 ảnh.")

if resume_source != OUTPUT_DIR:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    if plain_csv.is_file():
        shutil.copy2(plain_csv, resume_csv)
    else:
        with gzip.open(input_csv, "rb") as src, resume_csv.open("wb") as dst:
            shutil.copyfileobj(src, dst)
    shutil.copy2(resume_source / "checkpoint_config.json", resume_json)

MODEL_REVISION = resume_config["model_revision"]
print("Checkpoint source:", resume_source)
print("Checkpoint restored to:", OUTPUT_DIR)
print(f"Successful checkpoint pages: {resume_successes:,}; remaining: {9000 - resume_successes:,}")
print("Pinned model revision:", MODEL_REVISION)
print("Required Transformers:", resume_config.get("transformers_version"))
print("Required PyTorch:", resume_config.get("torch_version"))
del previous_checkpoint


## 4. Download and safely extract only **test images** and **test GT text**

Downloads `test/{text,nontext}/img.tar.gz` and `test/{text,nontext}/gt.tar.gz`, **not** train data, clean page images, or occluded annotations. The extraction flattens possible nested paths so that image and annotation basenames match. The first run may use several GB of disk and network bandwidth.

In [ ]:
def extract_flat(archive_path: str, destination: Path, allowed_suffix: str) -> int:
    """Safely copy only files of the specified type; never use archive-supplied paths."""
    destination.mkdir(parents=True, exist_ok=True)
    count = 0
    with tarfile.open(archive_path, mode="r:gz") as archive:
        for member in tqdm(archive, desc=f"Extract {destination.parent.name}/{destination.name}", leave=False):
            if not member.isfile() or not member.name.lower().endswith(allowed_suffix):
                continue
            # Ignore any member directory traversal; write only basenames.
            filename = Path(member.name).name
            target = destination / filename
            if not target.exists():
                stream = archive.extractfile(member)
                if stream is None:
                    continue
                with stream, target.open("wb") as out:
                    shutil.copyfileobj(stream, out)
            count += 1
    return count


def download_test_category(category: str):
    if category not in {"text", "nontext"}:
        raise ValueError(f"Unknown category: {category}")
    for kind, suffix in (("img", ".jpg"), ("gt", ".txt")):
        repo_path = f"test/{category}/{kind}.tar.gz"
        target_dir = DATA_ROOT / category / kind
        existing = list(target_dir.glob(f"*{suffix}")) if target_dir.exists() else []
        expected = 6000 if category == "text" else 3000
        if len(existing) >= expected:
            print(f"Reuse {category}/{kind}: {len(existing)} files")
            continue
        print(f"Downloading {repo_path} ...")
        tar_path = hf_hub_download(repo_id=REPO_ID, repo_type="dataset", filename=repo_path)
        n = extract_flat(tar_path, target_dir, suffix)
        print(f"  {category}/{kind}: {n:,} archive members of type {suffix} -> {target_dir}")


for category in CATEGORIES:
    download_test_category(category)

## 5. Pair input images with evaluation annotations

In [ ]:
def parse_occlusion_metadata(stem: str):
    # OPRB filename convention: OcclusionType__density__batchID... 
    parts = stem.split("__", 2)
    return {
        "occlusion_type": parts[0] if parts else "unknown",
        "occlusion_density": parts[1] if len(parts) > 1 else "unknown",
    }


def collect_test_pairs(categories):
    pairs = []
    for category in categories:
        image_dir = DATA_ROOT / category / "img"
        gt_dir = DATA_ROOT / category / "gt"
        images = {p.stem: p for p in image_dir.glob("*.jpg")}
        gts = {p.stem: p for p in gt_dir.glob("*.txt")}
        common = sorted(images.keys() & gts.keys())
        print(f"{category}: images={len(images):,}, gt={len(gts):,}, paired={len(common):,}")
        if not common:
            raise RuntimeError(f"No paired files for {category}. Check extraction and naming.")
        if MAX_SAMPLES_PER_CATEGORY is not None:
            rng = random.Random(RANDOM_SEED)
            common = sorted(rng.sample(common, min(MAX_SAMPLES_PER_CATEGORY, len(common))))
        for stem in common:
            pairs.append({
                "category": category,
                "sample_id": stem,
                "image_path": str(images[stem]),
                "gt_path": str(gts[stem]),
                **parse_occlusion_metadata(stem),
            })
    return pairs


pairs = collect_test_pairs(CATEGORIES)
print(f"Selected {len(pairs):,} samples. Set MAX_SAMPLES_PER_CATEGORY=None to use the entire test set.")
display(pd.DataFrame(pairs)[["category", "occlusion_type", "occlusion_density", "sample_id"]].head(8))

with Image.open(pairs[0]["image_path"]) as image:
    preview = image.copy()
preview.thumbnail((650, 850))
display(preview)

## 6. Read OPRB ground-truth text and define metrics

The OPRB GT archives contain 10-column DocBank rows: `token`, four bbox coordinates, three RGB values, `font`, `category`. The reader also accepts the 11-column layout described in the dataset card. We extract **only field 1 (`token`)**, preserving the source row order, and join tokens into a transcript. An empty transcript raises an error before model inference. Token ordering in a DocBank-derived annotation may differ from natural reading order on complex multi-column pages, so CER/WER should be interpreted with that limitation.

- **CER**: character-level edit distance divided by reference character count (lower is better).
- **WER**: word-level edit distance divided by reference word count (lower is better).
- **Token P/R/F1**: order-insensitive **bag-of-words** overlap, *not* bounding-box detection F1.
- Metrics can be **greater than 1.0** for very poor generations containing many insertions.

In [ ]:
def read_gt_tokens(gt_path: str):
    tokens = []
    with open(gt_path, "r", encoding="utf-8-sig", errors="replace") as file:
        for line in file:
            fields = line.rstrip("\r\n").split("\t")
            if len(fields) >= 10 and fields[0].strip():
                tokens.append(fields[0].strip())
    if not tokens:
        raise ValueError(f'No GT tokens read from {gt_path}. Check the annotation format and file contents.')
    return tokens


def clean_text(text: str) -> str:
    text = unicodedata.normalize("NFKC", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.casefold() if NORMALIZE_CASE else text


def bag_of_words_prf(reference: str, prediction: str):
    ref = Counter(reference.split())
    hyp = Counter(prediction.split())
    matched = sum((ref & hyp).values())
    ref_n, hyp_n = sum(ref.values()), sum(hyp.values())
    precision = matched / hyp_n if hyp_n else (1.0 if ref_n == 0 else 0.0)
    recall = matched / ref_n if ref_n else (1.0 if hyp_n == 0 else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def compute_metrics(ground_truth: str, prediction: str):
    ref, hyp = clean_text(ground_truth), clean_text(prediction)
    if not ref:
        raise ValueError("Empty GT transcript; cannot compute meaningful CER/WER.")
    p, r, f1 = bag_of_words_prf(ref, hyp)
    return {
        "cer": float(cer(ref, hyp)),
        "wer": float(wer(ref, hyp)),
        "token_precision": p,
        "token_recall": r,
        "token_f1": f1,
        "gt_words": len(ref.split()),
        "pred_words": len(hyp.split()),
    }


# Check metric code on a toy example BEFORE downloading the model.
assert compute_metrics("The cat sat", "The cat")['wer'] == 1/3
print('Toy check:', compute_metrics("The cat sat", "The cat"))
print('First GT tokens:', read_gt_tokens(pairs[0]['gt_path'])[:35])

## 7. Load Florence-2-base in float16

Use the native Transformers `Florence2ForConditionalGeneration` class and the converted checkpoint recommended in the Transformers documentation. No remote Python model code or 4-bit quantization is required. Float16 is supported on a Tesla T4.

The cell releases any previously loaded model before loading Florence. The model remains pretrained; no parameters are updated. Its checkpoint image processor controls resolution.


In [ ]:
from transformers import AutoProcessor, Florence2ForConditionalGeneration

# Release the previous model when switching inside the same notebook kernel.
for object_name in ('model', 'processor'):
    globals().pop(object_name, None)
gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(MODEL_ID, revision=MODEL_REVISION)
model = Florence2ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    dtype=torch.float16,
    attn_implementation="sdpa",
).to("cuda")
model.eval()

decoder_limit = int(model.config.text_config.max_position_embeddings)
assert 0 < MAX_NEW_TOKENS <= decoder_limit, (
    f"MAX_NEW_TOKENS must be between 1 and {decoder_limit} for this checkpoint."
)
# Use the published checkpoint decoding defaults for this baseline.
# The n-gram guard blocks repeated token trigrams, including some legitimate ones.
model.generation_config.num_beams = NUM_BEAMS
model.generation_config.early_stopping = EARLY_STOPPING
model.generation_config.no_repeat_ngram_size = NO_REPEAT_NGRAM_SIZE

IMAGE_PREPROCESSING = processor.image_processor.to_dict()
print('Model:', MODEL_ID)
print('Parameters:', f'{sum(p.numel() for p in model.parameters()):,}')
print('Precision: float16; no fine-tuning')
print('Processor image size:', processor.image_processor.size)
print('Decoder position limit:', decoder_limit)
print(f'Allocated GPU memory: {torch.cuda.memory_allocated() / 2**30:.2f} GiB')


## 8. Run OCR on a single occluded image

Florence takes the task token `<OCR>` instead of a chat prompt. It is an encoder-decoder model: decode its entire output sequence. No input-prompt prefix is removed. The processor removes task markers from the transcription.

This example is also a GPU warm-up; the per-image timing reported in Section 9 is measured after it.


In [ ]:
OCR_PROMPT = "<OCR>"


@torch.inference_mode()
def transcribe_page(image_path: str):
    """Pass only an occluded image and the fixed OCR task to Florence."""
    torch.cuda.synchronize()
    started = time.perf_counter()
    with Image.open(image_path) as source:
        image = source.convert("RGB")

    # PIL RGB becomes HWC. Explicit layout also handles 1x1 RGB images;
    # auto-inference otherwise mistakes their height for the channel count.
    inputs = processor(
        text=OCR_PROMPT, images=image, return_tensors="pt",
        images_kwargs={"input_data_format": "channels_last"},
    ).to("cuda", torch.float16)
    output_ids = model.generate(
        **inputs,
        max_new_tokens=MAX_NEW_TOKENS,
        num_beams=NUM_BEAMS,
        do_sample=False,
        use_cache=True,
        no_repeat_ngram_size=NO_REPEAT_NGRAM_SIZE,
        early_stopping=EARLY_STOPPING,
    )
    raw_text = processor.batch_decode(
        output_ids, skip_special_tokens=False
    )[0]
    parsed = processor.post_process_generation(
        raw_text, task=OCR_PROMPT, image_size=image.size
    )
    prediction = parsed[OCR_PROMPT].strip()
    # The sequence starts with one decoder-start token, not the input prompt.
    generated_tokens = int(output_ids.shape[-1] - 1)
    possibly_truncated = generated_tokens >= MAX_NEW_TOKENS
    torch.cuda.synchronize()
    elapsed_sec = time.perf_counter() - started
    del inputs, output_ids
    return prediction, elapsed_sec, generated_tokens, possibly_truncated


sample = pairs[0]
example_gt = ' '.join(read_gt_tokens(sample['gt_path']))
example_prediction, example_time, example_tokens, example_truncated = transcribe_page(sample['image_path'])
print('CATEGORY:', sample['category'])
print('IMAGE:', Path(sample['image_path']).name)
print('LATENCY (s):', round(example_time, 2))
print('GENERATED TOKENS:', example_tokens, 'Possible truncation:', example_truncated)
print('GT (first 700 chars):', example_gt[:700])
print('PREDICTION (first 700 chars):', example_prediction[:700])
print('METRICS:', compute_metrics(example_gt, example_prediction))


## 9. Evaluate selected test pages and save checkpoints

Each attempt is saved atomically to `per_image_results.csv`. Resume requires the same model, prompt, decoding, preprocessing, and scoring settings; `checkpoint_config.json` enforces this. Increasing the sample limit from 5 to `None` can reuse successful predictions. Failed pages are retried on the next run, and category plus sample ID identifies a page.

The loop stops after `MAX_EVAL_HOURS` so Sections 10–12 can summarize and export partial results. A partial run is not a completed benchmark: check the remaining-page count. Back up the CSV and checkpoint settings before ending a session. Restore both before resuming in a fresh session.

**Live logs:** Section 9 prints a flushed `PROGRESS` line after its first attempt, approximately every 60 seconds thereafter, and after the last attempt. It includes successful pages, attempts, errors, elapsed time, average wall time per page, and an ETA for unattempted pages. These lines appear in a saved Kaggle version’s live logs without notebook widgets. Editing this file does not update a version already running.


In [ ]:
from importlib.metadata import version
import csv
import sys

csv.field_size_limit(min(sys.maxsize, 2**31 - 1))

RESULTS_CSV = OUTPUT_DIR / 'per_image_results.csv'
CHECKPOINT_CONFIG = OUTPUT_DIR / 'checkpoint_config.json'
checkpoint_settings = {
    'dataset': REPO_ID,
    'split': 'test',
    'model': MODEL_ID,
    'model_revision': getattr(model.config, '_commit_hash', None),
    'precision': 'float16',
    'prompt': OCR_PROMPT,
    'image_preprocessing': IMAGE_PREPROCESSING,
    'max_new_tokens': MAX_NEW_TOKENS,
    'num_beams': NUM_BEAMS,
    'do_sample': False,
    'no_repeat_ngram_size': NO_REPEAT_NGRAM_SIZE,
    'early_stopping': EARLY_STOPPING,
    'normalize_case': NORMALIZE_CASE,
    'metric_definition': 'Macro means of per-image CER/WER and bag-of-words overlap',
    'transformers_version': version('transformers'),
    'torch_version': str(torch.__version__),
}
if CHECKPOINT_CONFIG.exists():
    saved_settings = json.loads(CHECKPOINT_CONFIG.read_text(encoding='utf-8'))
    if saved_settings != checkpoint_settings:
        raise ValueError('Checkpoint settings differ. Choose a new OUTPUT_DIR for this experiment.')
elif RESULTS_CSV.exists():
    raise ValueError('Checkpoint CSV has no checkpoint_config.json. Restore both or choose a new OUTPUT_DIR.')
else:
    CHECKPOINT_CONFIG.write_text(json.dumps(checkpoint_settings, indent=2), encoding='utf-8')


def sample_key(record):
    return record['category'], record['sample_id']


selected_keys = {sample_key(sample) for sample in pairs}
if RESULTS_CSV.exists():
    previous = pd.read_csv(RESULTS_CSV, engine="python", keep_default_na=False).to_dict(orient='records')
    if any(record.get('model_id') != MODEL_ID for record in previous):
        raise ValueError('This CSV contains results from another model. Use a separate OUTPUT_DIR.')
    saved_rows = {sample_key(record): record for record in previous}
else:
    saved_rows = {}

finished = {key for key, record in saved_rows.items() if record.get('status') == 'ok'}
pending = [sample for sample in pairs if sample_key(sample) not in finished]
print(f'Reusing {len(finished & selected_keys):,} successful selected pages; {len(pending):,} remaining.', flush=True)
evaluation_started = time.perf_counter()
last_progress_log = evaluation_started
successful_count = len(finished & selected_keys)
errors_this_run = 0


def format_duration(seconds):
    hours, remainder = divmod(int(seconds), 3600)
    minutes, seconds = divmod(remainder, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}'


for attempted_this_run, sample in enumerate(tqdm(pending, desc='Florence OCR evaluation', unit='page', mininterval=60), start=1):
    if MAX_EVAL_HOURS is not None and time.perf_counter() - evaluation_started >= MAX_EVAL_HOURS * 3600:
        print('Time budget reached. Export this checkpoint, then resume later.', flush=True)
        break
    result = dict(sample)
    result['model_id'] = MODEL_ID
    try:
        ground_truth = ' '.join(read_gt_tokens(sample['gt_path']))
        prediction, elapsed, n_tokens, truncated = transcribe_page(sample['image_path'])
        result.update({
            'status': 'ok',
            'prediction': prediction,
            'ground_truth': ground_truth,
            'latency_sec': elapsed,
            'generated_tokens': n_tokens,
            'possibly_truncated': truncated,
            **compute_metrics(ground_truth, prediction),
        })
    except Exception as exc:
        result.update({'status': 'error', 'error': repr(exc)})
        print('Failure:', sample['sample_id'], repr(exc))
        gc.collect()
        torch.cuda.empty_cache()
    saved_rows[sample_key(sample)] = result
    # Keep previous successful pages even if the current selection is smaller.
    temporary_csv = RESULTS_CSV.with_suffix('.csv.tmp')
    pd.DataFrame(saved_rows.values()).to_csv(temporary_csv, index=False)
    temporary_csv.replace(RESULTS_CSV)
    successful_count += result['status'] == 'ok'
    errors_this_run += result['status'] == 'error'
    now = time.perf_counter()
    if attempted_this_run == 1 or now - last_progress_log >= 60 or attempted_this_run == len(pending):
        elapsed = now - evaluation_started
        average_sec = elapsed / attempted_this_run
        unattempted = len(pending) - attempted_this_run
        print(
            f'PROGRESS: {successful_count:,}/{len(pairs):,} successful ({100 * successful_count / len(pairs):.1f}%) | '
            f'attempted this run={attempted_this_run:,}/{len(pending):,} | errors this run={errors_this_run:,} | '
            f'elapsed={format_duration(elapsed)} | avg={average_sec:.2f}s/page | '
            f'ETA for unattempted pages={format_duration(unattempted * average_sec)}',
            flush=True,
        )
        last_progress_log = now

rows = [record for key, record in saved_rows.items() if key in selected_keys]
results = pd.DataFrame(rows)
if results.empty:
    raise RuntimeError('No pages were attempted. Check MAX_EVAL_HOURS and the checkpoint.')
print('Saved:', RESULTS_CSV)
print('Successful:', (results.status == 'ok').sum(), '/', len(pairs), 'selected pages')
display(results.reindex(columns=[
    'category', 'occlusion_type', 'cer', 'wer', 'token_f1',
    'latency_sec', 'possibly_truncated', 'status', 'error'
]).head(10))


## 10. Summarize overall results and compare occlusion categories/types

In [ ]:
import numpy as np

ok = results.loc[results['status'] == 'ok'].copy()
assert len(ok), 'No successful predictions. Inspect the error column.'

metric_cols = ['cer', 'wer', 'token_precision', 'token_recall', 'token_f1', 'latency_sec']
# keep_default_na=False preserves GT/prediction text, but numeric CSV columns
# containing blank error rows may become strings. Resumed rows and new rows
# therefore need explicit numeric types before means/grouped aggregations.
for column in metric_cols:
    ok[column] = pd.to_numeric(ok[column], errors='raise').astype(float)
if not np.isfinite(ok[metric_cols].to_numpy(dtype=float)).all():
    raise ValueError('Successful rows contain missing or non-finite metrics.')

# bool('False') is True: parse saved CSV flags explicitly instead.
flag_values = ok['possibly_truncated'].astype(str).str.strip().str.casefold()
flag_map = {'true': True, 'false': False, '1': True, '0': False, '1.0': True, '0.0': False}
if not flag_values.isin(flag_map).all():
    raise ValueError(f'Invalid truncation flags: {flag_values[~flag_values.isin(flag_map)].unique().tolist()}')
ok['possibly_truncated'] = flag_values.map(flag_map).astype(bool)

def aggregate_metrics(df, group_by):
    summary = df.groupby(group_by, dropna=False).agg(
        n=('sample_id', 'count'),
        **{f'mean_{c}': (c, 'mean') for c in metric_cols},
        truncated=('possibly_truncated', 'sum'),
    ).reset_index()
    return summary

# IMPORTANT: These are macro averages across pages, NOT corpus WER/CER.
overall = {f'mean_{m}': float(ok[m].mean()) for m in metric_cols}
overall.update({
    'n_selected': int(len(pairs)),
    'n_success': int(len(ok)),
    'n_remaining': int(len(pairs) - len(ok)),
    'evaluation_complete': bool(len(ok) == len(pairs)),
    'n_errors': int(len(results) - len(ok)),
    'n_possibly_truncated': int(ok['possibly_truncated'].sum()),
})

by_category = aggregate_metrics(ok, ['category'])
by_type = aggregate_metrics(ok, ['category', 'occlusion_type'])
by_density = aggregate_metrics(ok, ['category', 'occlusion_density'])

print('OVERALL macro-averaged metrics:')
print(json.dumps(overall, indent=2))
print('\nBY OCCLUSION REGION:')
display(by_category)
print('\nBY OCCLUSION TYPE:')
display(by_type)
print('\nBY OCCLUSION DENSITY:')
display(by_density)

by_category.to_csv(OUTPUT_DIR / 'summary_by_category.csv', index=False)
by_type.to_csv(OUTPUT_DIR / 'summary_by_occlusion_type.csv', index=False)
by_density.to_csv(OUTPUT_DIR / 'summary_by_density.csv', index=False)

# A rough full-test estimate; a 10-page trial has substantial uncertainty.
region_times = by_category.set_index('category')['mean_latency_sec']
if {'text', 'nontext'}.issubset(region_times.index):
    estimated_full_hours = (6000 * region_times['text'] + 3000 * region_times['nontext']) / 3600
    print(f'Estimated inference time for all 9,000 pages: {estimated_full_hours:.1f} hours.')
print('Pages still requiring a successful evaluation:', overall['n_remaining'])


## 11. Visualize the errors

In [ ]:
ax = by_category.set_index('category')[['mean_cer', 'mean_wer']].plot(
    kind='bar', figsize=(7, 4), rot=0, title='OPRB test: mean CER and WER by region'
)
ax.set_xlabel('Occlusion region')
ax.set_ylabel('Error rate (lower is better)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'cer_wer_by_category.png', dpi=150)
plt.show()

ax = by_type.set_index('occlusion_type').groupby(level=0)['mean_wer'].mean().sort_values().plot(
    kind='barh', figsize=(9, 5), title='Mean WER by occlusion type (unweighted category average)'
)
ax.set_xlabel('Mean word error rate (lower is better)')
plt.tight_layout()
plt.savefig(OUTPUT_DIR / 'wer_by_occlusion_type.png', dpi=150)
plt.show()

## 12. Export results (.zip) and reproducibility settings

In [ ]:
from importlib.metadata import version

run_config = {
    **checkpoint_settings,
    'input_channel_layout': 'channels_last',
    'input_layout_note': 'Explicit HWC for PIL RGB, including 1x1 images; processor configuration and generation settings unchanged.',
    'categories': CATEGORIES,
    'no_finetuning': True,
    'max_samples_per_category': MAX_SAMPLES_PER_CATEGORY,
    'random_seed': RANDOM_SEED,
    'max_eval_hours_per_run': MAX_EVAL_HOURS,
    'metrics': overall,
    'gpu_name': torch.cuda.get_device_name(0),
    'jiwer_version': version('jiwer'),
}
(OUTPUT_DIR / 'run_config.json').write_text(json.dumps(run_config, indent=2), encoding='utf-8')
zip_name = shutil.make_archive(str(OUTPUT_DIR), 'zip', root_dir=OUTPUT_DIR)
print('Results ZIP:', zip_name)
print('Download the ZIP from Kaggle output files.')
print('For a fresh session, restore per_image_results.csv AND checkpoint_config.json to OUTPUT_DIR.')
if not overall['evaluation_complete']:
    print('This export is partial. Resume to finish all selected pages.')


## Reporting checklist and limitations

This notebook provides one model's results. The assignment requires a comparative report for **at least five pretrained lightweight models**, under **four A4 pages in PDF format**, with Introduction, Dataset Statistics, Model Descriptions, Experimental Results, and Conclusion.

Report the exact model checkpoint, parameter count, float16 precision, GPU, processor image size, `<OCR>` task, 3-beam decoding, no-repeat token trigram rule, early stopping, 1,024-token cap, evaluated-page count, success/failure count, mean CER/WER, token precision/recall/F1, average latency, and separate `text`/`nontext` results. Compare occlusion types, show correct and incorrect examples, and discuss accuracy/size/time trade-offs. Use the same evaluation images and scoring rules for every model. Document each model's preprocessing and decoding settings.

1. Five pages per category is a trial, not a representative final comparison. Set `MAX_SAMPLES_PER_CATEGORY = None` for the full test set, or use a common representative subset only if permitted by your instructor.
2. A time-limited export can contain a partial evaluation. Check `evaluation_complete`, `n_selected`, `n_success`, and `n_remaining` in `run_config.json` before reporting a completed experiment.
3. This is page transcription against clean-page text, not image restoration or localization. Hidden tokens can increase error rates. Ground truth is never provided to the model.
4. Annotation row order approximates reading order. Multi-column layouts, `(cid:...)` artifacts, equations, and differing math notation can increase CER/WER independently of occlusion recognition.
5. The processor's default resizing and the decoder's 1,024-token capacity can reduce text recovery. Inspect `possibly_truncated`; do not exceed the checkpoint's learned decoder-position limit. The initial single-page example warms the GPU before benchmark timing.
6. Token precision/recall/F1 are bag-of-words overlap; CER/WER are macro means across pages. CER/WER can exceed 1.0 when predictions contain many insertions.
7. Use a separate output directory if changing the model, decoding, image preprocessing, or scoring. The checkpoint guard prevents mixing different experiments. Successful pages are reused; failed pages are retried on the next evaluation run.

Sources: [OPRB dataset](https://huggingface.co/datasets/kpurkayastha/OPRB), [Florence-2-base converted checkpoint](https://huggingface.co/florence-community/Florence-2-base), [original Microsoft model](https://huggingface.co/microsoft/Florence-2-base), [Transformers implementation guide](https://huggingface.co/docs/transformers/v4.57.1/en/model_doc/florence2).
